# churn_predict - slim runtime artifact

Builds `models/artifacts_slim.pkl` for the Streamlit Community Cloud deployment.

The shipped artifact is 246 MB, and **234 MB of that is a fitted `RandomSurvivalForest`** that
`dashboard.py` never calls (it only reads `survival["probs"]`, the two c-indices, `horizons`
and `corr_s90_binary`). Loading it costs ~684 MB of peak RSS on a 690 MB container.

Input is the exact artifact the dashboard was verified against, mounted as the private Kaggle
dataset `vivekkopthsd/churn-predict-full-artifact`. The Hugging Face copy is **not** used: it is
stale (scikit-learn 1.9.0, ROAS 22.0x instead of 15.9x) and would change the displayed numbers.

This notebook prunes to ~10 MB, asserts the contract with `dashboard.py`, re-solves the MILP
with the sparse formulation, and measures cold-start memory in a fresh process.

Output: `/kaggle/working/artifacts_slim.pkl` -> commit as `models/artifacts_slim.pkl`.

In [ ]:
# Pinned to the versions the artifact was built with, so unpickling is exact.
!pip install -q "scikit-learn==1.9.1" "scikit-survival==0.28.0" "xgboost==3.2.*" psutil

import sklearn, sksurv, xgboost, numpy, pandas
print("sklearn", sklearn.__version__, "| sksurv", sksurv.__version__,
      "| xgboost", xgboost.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)

In [ ]:
import os, glob, pickle, hashlib, re, warnings
warnings.filterwarnings("ignore")

# Canonical source: the exact artifact the dashboard was verified against
# (246,580,712 bytes, pickled with scikit-learn 1.9.1). It ships as a private Kaggle
# dataset because the Hugging Face copy is stale: sklearn 1.9.0 and a different optimum
# (ROAS 22.0x instead of 15.9x), which would silently change the displayed numbers.
MOUNT = "/kaggle/input/churn-predict-full-artifact/artifacts.pkl"
CANDIDATES = [MOUNT] if os.path.exists(MOUNT) else []     + [p for p in glob.glob("/kaggle/input/**/artifacts.pkl", recursive=True)
       if "slim" not in os.path.basename(p)]
assert CANDIDATES, "attach dataset vivekkopthsd/churn-predict-full-artifact to this kernel"
FULL = CANDIDATES[0]
print(f"source artifact: {FULL}  {os.path.getsize(FULL)/1e6:.1f} MB")

with open(FULL, "rb") as f:
    head = f.read(8_000_000)
m = re.search(rb"_sklearn_version.{0,3}(\d+\.\d+\.\d+)", head, re.S)
BUILT_WITH = m.group(1).decode() if m else "unknown"
print("pickled with scikit-learn", BUILT_WITH)
assert BUILT_WITH == "1.9.1", f"artifact was built with {BUILT_WITH}, expected 1.9.1"

with open(FULL, "rb") as f:
    A = pickle.load(f)

# Objects dashboard.py never reads. survival["rsf"] alone is ~234 MB of the file.
DROP = ("tr", "te", "pre", "pre_s", "pca6", "kmeans", "prob_te",
        "segment_profiles", "seg_names", "silhouette")
SURVIVAL_DROP = ("rsf", "cox")

per_key = {k: len(pickle.dumps(v, protocol=4)) / 1e6 for k, v in A.items()}
print("\nlargest keys in the full artifact (MB):")
for k, mb in sorted(per_key.items(), key=lambda r: -r[1])[:4]:
    print(f"  {k:14s} {mb:8.1f}")
print(f"  {'survival.rsf':14s} {len(pickle.dumps(A['survival']['rsf'], protocol=4))/1e6:8.1f}")

slim = {k: v for k, v in A.items() if k not in DROP}
slim["survival"] = {k: v for k, v in A["survival"].items() if k not in SURVIVAL_DROP}

OUT = "/kaggle/working/artifacts_slim.pkl"
with open(OUT, "wb") as f:
    pickle.dump(slim, f, protocol=4)

print(f"\nslim artifact: {os.path.getsize(OUT)/1e6:.1f} MB "
      f"({os.path.getsize(OUT)/os.path.getsize(FULL)*100:.1f}% of full)")
print("sha256:", hashlib.sha256(open(OUT, "rb").read()).hexdigest())
print("keys:", sorted(slim.keys()))

In [ ]:
# Contract check: everything dashboard.py reads must survive the prune.
READS = ["churn_metrics_df", "churn_summary", "cust", "feature_names", "fitted_models",
         "offer_result", "perm_imp", "rf_imp", "sensitivity_df", "shap_importance",
         "survival", "test_scores"]
missing = [k for k in READS if k not in slim]
assert not missing, f"missing keys: {missing}"
assert set(slim["survival"]) == {"cindex_cox", "cindex_rsf", "corr_s90_binary", "horizons", "probs"}
print("all 12 dashboard keys present | survival keys:", sorted(slim["survival"]))
print("fitted models:", list(slim["fitted_models"]))
print("milp budgets precomputed:", len(slim["offer_result"]["milp_by_budget"]))
print("S_90 shape:", slim["survival"]["probs"].shape)

# The displayed figures must not move. Expected: $159,027 retained, ROAS 15.90x, 400 customers.
pre = slim["offer_result"]["milp_by_budget"][10000]
print(f"precomputed @ $10,000: rev=${pre['rev']:,.0f} roas={pre['roas']:.2f}x "
      f"customers={pre['customers']} cost=${pre['cost']:,.0f}")

# Re-solve with the sparse formulation the patched dashboard uses (Kronecker identity)
# and prove it reproduces the stored optimum.
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import csr_matrix, eye as speye, kron

ts = slim["test_scores"]
n = len(ts)
offers = np.array([5, 10, 15, 20, 25], dtype=float)
p, V = ts["Churn_Prob"].to_numpy(), ts["CLV_3Yr"].to_numpy()
r = lambda d: np.clip(0.04 + 0.018 * d, 0, 0.60)
DV = np.array([[pi * Vi * (r(d) - r(0)) for d in offers] for pi, Vi in zip(p, V)])
A_one = kron(speye(n, format="csr"), csr_matrix(np.ones((1, len(offers)))))
res = milp(c=-DV.ravel(), integrality=np.ones(n * len(offers)), bounds=Bounds(0, 1),
           constraints=[LinearConstraint(np.tile(offers, n), ub=10000),
                        LinearConstraint(A_one, ub=1.0)], options={"time_limit": 60})
x = np.round(res.x).astype(int).reshape(n, len(offers))
offer, incr = (x * offers).sum(1), (x * DV).sum(1)
keep = offer > 0
retained, roas, customers = incr[keep].sum(), incr[keep].sum()/offer[keep].sum(), int(keep.sum())
print(f"MILP optimal={res.status == 0} retained=${retained:,.0f} roas={roas:.2f}x "
      f"customers={customers} matrix={A_one.data.nbytes/1e6:.3f} MB")

assert res.status == 0, "MILP did not reach optimality"
assert abs(retained - 159027) < 5, f"retained moved: {retained:,.2f}"
assert abs(roas - 15.90) < 0.05, f"ROAS moved: {roas:.3f}"
assert customers == 400, f"offer count moved: {customers}"
assert abs(pre["rev"] - retained) < 5, "slim artifact disagrees with the live solve"
print("\nOK - displayed figures unchanged, sparse matrix is", 
      f"{922e6/A_one.data.nbytes:.0f}x smaller than the dense build (922 MB).")

In [ ]:
# Cold-start cost of the slim artifact in a fresh process (imports + unpickle).
script = r"""
import time, pickle, warnings, resource
warnings.filterwarnings("ignore")
import numpy, pandas, sklearn, sksurv, xgboost
rss = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
print(f"after imports  {rss():6.0f} MB")
t0 = time.time()
A = pickle.load(open("/kaggle/working/artifacts_slim.pkl", "rb"))
print(f"load {time.time()-t0:.2f}s  peak {rss():6.0f} MB  keys {len(A)}")
"""
open("/kaggle/working/_coldstart.py", "w").write(script)
!python /kaggle/working/_coldstart.py

## Optional: rebuild from scratch instead of pruning

Run this only if you want the full pipeline re-executed rather than the existing models
pruned. It needs the repo's `data/*.csv` and writes both artifacts via `train_models.main()`:

```bash
git clone --depth 1 https://github.com/bravo2024/churn_predict.git repo
cd repo && pip install -q -r requirements.txt && python train_models.py
# -> models/artifacts.pkl and models/artifacts_slim.pkl
```

That path retrains RF/GB/XGB/RSF (~10-15 min on a Kaggle CPU) and must run with
`scikit-learn==1.9.1` so the pickle matches the Cloud environment.